# ♻️ Notebook 2: Refactoring to SOLID

## 🛠️ Setup

```bash
cd 07-object-oriented-design/oo-analysis-and-design
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


We'll refactor a **deliberately bad** `Order` class to follow SOLID, step by step.

In [ ]:
# ❌ Everything-in-one-class
class BadOrder:
    def __init__(self, items, user_email):
        self.items = items
        self.user_email = user_email

    def total(self):
        return sum(p for _, p in self.items)

    def charge_stripe(self, card_number):
        print(f"hardcoded stripe call with {card_number} for ${self.total()}")
        return "ok"

    def send_confirmation(self):
        print(f"SMTP → {self.user_email}: your order total was ${self.total()}")

    def save(self):
        print(f"raw SQL INSERT for {self.items}")

o = BadOrder([("book", 10), ("pen", 2)], "alice@example.com")
o.charge_stripe("4242")
o.send_confirmation()
o.save()


### Problems
- **SRP**: billing + email + DB + domain in one class.
- **OCP**: new payment provider? Edit this file.
- **DIP**: hardcoded to Stripe and SMTP.

### Refactor

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

# Domain — pure data, one job
@dataclass
class Order:
    items: list[tuple[str, float]]
    user_email: str
    def total(self) -> float:
        return sum(p for _, p in self.items)

# Abstractions (interfaces)
class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount: float, token: str) -> str: ...

class Notifier(ABC):
    @abstractmethod
    def send(self, to: str, message: str) -> None: ...

class OrderRepo(ABC):
    @abstractmethod
    def save(self, order: Order) -> None: ...


In [ ]:
# Concrete implementations (easy to swap)
class StripeGateway(PaymentGateway):
    def charge(self, amount, token):
        print(f"  [stripe] charged ${amount} with {token}")
        return "ok"

class PayPalGateway(PaymentGateway):
    def charge(self, amount, token):
        print(f"  [paypal] charged ${amount} with {token}")
        return "ok"

class EmailNotifier(Notifier):
    def send(self, to, message):
        print(f"  [email→{to}] {message}")

class InMemoryRepo(OrderRepo):
    def __init__(self): self.store = []
    def save(self, order): self.store.append(order); print("  [db] saved")

# OrderService orchestrates — dependencies injected (DIP)
class OrderService:
    def __init__(self, repo: OrderRepo, gateway: PaymentGateway, notifier: Notifier):
        self.repo = repo
        self.gateway = gateway
        self.notifier = notifier

    def place(self, order: Order, payment_token: str) -> None:
        self.gateway.charge(order.total(), payment_token)
        self.repo.save(order)
        self.notifier.send(order.user_email, f"Your total: ${order.total()}")

svc = OrderService(InMemoryRepo(), StripeGateway(), EmailNotifier())
svc.place(Order([("book", 10), ("pen", 2)], "alice@example.com"), "tok_test")

# Switching payment provider = one line change
svc2 = OrderService(InMemoryRepo(), PayPalGateway(), EmailNotifier())
svc2.place(Order([("hat", 20)], "bob@example.com"), "tok_test")


### What changed
- **SRP**: `Order` is just data; `OrderService` orchestrates; `Repo`/`Gateway`/`Notifier` each do one job.
- **OCP**: Added PayPal by writing a new class, not editing existing code.
- **DIP**: `OrderService` depends on abstract interfaces.
- **Tests**: We can pass a `FakeGateway` and `FakeNotifier` without hitting Stripe or SMTP.

This is the shape most production services take.